# ARG Dashboard V2 BACI Pipeline

This notebook rebuilds the dashboard intermediate files from CEPII BACI HS92 trade flows. Product names, sectors, and green-product flags continue to come from the existing HS92 reference files.

In [ ]:
from pathlib import Path
import os

FOCUS_ISO = 'ARG'
FOCUS_COUNTRY_NAME = 'Argentina'
BACI_VERSION = 'V202601'
YEARS = [2020, 2021, 2022, 2023, 2024]

project_root = Path.cwd()
if project_root.name == 'code':
    project_root = project_root.parent
os.chdir(project_root)

print(project_root)


## Source Separation

- Trade flows: BACI HS92, variables `t`, `i`, `j`, `k`, `v`, `q`; `v` is converted from thousand USD to USD.
- Product labels and sectors: existing `data/input/hs92_4digits.csv` and `data/input/product_hs92.csv`.
- Country filter, GDP weights, and distance auxiliaries: existing dashboard inputs/intermediates.

In [ ]:
import pandas as pd

baci_dir = project_root / 'data' / 'input' / 'BACI_HS92_V202601'
country_codes = pd.read_csv(baci_dir / 'country_codes_V202601.csv')
country_codes[country_codes['country_iso3'].eq(FOCUS_ISO)]


## Build Intermediates

The builder creates a BACI-derived HS4 bilateral trade file compatible with the app schema, then writes the complexity, potential-market, and opportunity-metric CSVs used by Streamlit.

In [ ]:
import sys
sys.path.insert(0, str(project_root / 'code'))
from build_baci_v2 import main

main()


## Sanity Check: 9999 Share

BACI should not concentrate Argentina exports under HS4 `9999`; the generated BACI-compatible input should therefore have a zero or near-zero `9999` share for Argentina.

In [ ]:
trade = pd.read_csv(
    project_root / 'data' / 'input' / 'hs92_country_country_product_year_6_2020_2024.csv',
    usecols=['country_iso3_code', 'product_hs92_code', 'year', 'export_value'],
    dtype={'country_iso3_code': 'string', 'product_hs92_code': 'string'},
)
arg = trade[trade['country_iso3_code'].eq(FOCUS_ISO)].copy()
arg['hs4'] = arg['product_hs92_code'].astype('string').str.zfill(4).str[:4]
summary = arg.groupby('year').agg(
    total=('export_value', 'sum'),
    hs4_9999=('export_value', lambda s: s[arg.loc[s.index, 'hs4'].eq('9999')].sum()),
)
summary['hs4_9999_share'] = summary['hs4_9999'] / summary['total']
summary
